# 03 — The √t Law

In [`02_many_paths/`](../02_many_paths/) we noticed something: the standard deviation of where 500 walks ended up at $t = 1000$ was almost exactly $\sqrt{1000} \approx 31.6$. We saw this as a side observation. In this notebook we put it center stage.

The claim:

> **For a ±1 random walk starting at 0, the standard deviation of position at time $t$ is exactly $\sqrt{t}$.**

Three things to do:

1. **Measure it** — run thousands of walks, compute the std at *every* time, see if it really is $\sqrt{t}$ throughout.
2. **Derive it** — two lines of probability that explain *why* the answer is √t and not, say, $t$, or $\log t$.
3. **Visualize it** — including the picture that makes the deepest version of this fact unforgettable: rescaling by $1 / \sqrt{t}$ collapses the entire distribution onto a single shape.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

## Step 1: measure the std at every time

We'll generate a big batch of walks (10,000 of them) and compute the standard deviation across all walks at *each* time step. That gives us an empirical curve $\hat{\sigma}(t)$ to compare to the theoretical $\sqrt{t}$.

In [ ]:
N_WALKS = 10_000
N_STEPS = 1000
SEED = None
rng = np.random.default_rng(SEED)
steps = rng.choice([-1, 1], size=(N_WALKS, N_STEPS))
positions = np.concatenate(
    [np.zeros((N_WALKS, 1), dtype=int), np.cumsum(steps, axis=1)],
    axis=1,
)

# std across walks (axis=0) at each time → shape (N_STEPS + 1,)
empirical_std = positions.std(axis=0)
empirical_var = positions.var(axis=0)
t = np.arange(N_STEPS + 1)

print("empirical std vs sqrt(t) at a few checkpoints:")
print(f"{'t':>6} {'empirical std':>16} {'sqrt(t)':>12} {'ratio':>10}")
for t_check in [1, 10, 100, 500, 1000]:
    s = empirical_std[t_check]
    sq = np.sqrt(t_check)
    print(f"{t_check:>6} {s:>16.4f} {sq:>12.4f} {s/sq:>10.4f}")

The ratios are all very close to 1.0. With 10,000 walks per measurement, sampling noise is tiny — what we're seeing is a real equality, not a coincidence. Now let's plot the whole curve.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t, empirical_std, label=f"empirical std ({N_WALKS:,} walks)", linewidth=2.5)
ax.plot(t, np.sqrt(t), label=r"$\sqrt{t}$", linestyle="--", color="red", linewidth=2)
ax.set_xlabel("step number (t)")
ax.set_ylabel("standard deviation of position")
ax.set_title("the empirical std curve sits exactly on $\\sqrt{t}$")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

The two curves are visually indistinguishable. So $\hat{\sigma}(t) \approx \sqrt{t}$ holds at every $t$, not just at the endpoint.

## Step 2: derive it

Why √t and not something else? The proof is two lines, but they rely on one of the most useful facts in probability.

**Fact:** if $X$ and $Y$ are *independent* random variables, then $\text{Var}(X + Y) = \text{Var}(X) + \text{Var}(Y)$. **Variances of independent things add.**

(For *means*, this works without independence: $\mathbb{E}[X + Y] = \mathbb{E}[X] + \mathbb{E}[Y]$ always. The independence requirement on variances is the subtle part.)

Now, our random walk position at time $t$ is just the sum of $t$ independent ±1 steps:

$$X_t = \text{step}_1 + \text{step}_2 + \cdots + \text{step}_t$$

Each step has variance $\text{Var}(\pm 1) = \mathbb{E}[\text{step}^2] - (\mathbb{E}[\text{step}])^2 = 1 - 0 = 1$. Since the steps are independent:

$$\text{Var}(X_t) = \underbrace{1 + 1 + \cdots + 1}_{t \text{ terms}} = t$$

Take the square root, and there's the answer:

$$\sigma(X_t) = \sqrt{t}$$

That's the whole proof. The variance grows *linearly* in time because variances add. The std is the square root of the variance, so the std grows as the *square root* of time.

### Confirming visually: variance is linear in t

If the proof is right, plotting variance (not std) against $t$ should give a perfectly straight line through the origin with slope 1.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t, empirical_var, label="empirical variance", linewidth=2.5)
ax.plot(t, t, label=r"$t$", linestyle="--", color="red", linewidth=2)
ax.set_xlabel("step number (t)")
ax.set_ylabel("variance of position")
ax.set_title("variance grows linearly in t — that's the underlying fact")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

A perfectly straight line. **The √t scaling of the std isn't a coincidence — it's a direct consequence of the fact that independent variances add.**

## Step 3: visualize the law from a few angles

### Angle 1: log-log plot — slope 1/2

If $y = \sqrt{t}$, then $\log y = \tfrac{1}{2} \log t$ — a straight line with slope $\tfrac{1}{2}$ on log-log axes. That's the standard physicist's check for power-law scaling.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(t[1:], empirical_std[1:], label="empirical std", linewidth=2.5)
ax.loglog(t[1:], np.sqrt(t[1:]), label=r"$\sqrt{t}$", linestyle="--", color="red", linewidth=2)
ax.set_xlabel("step number (t, log scale)")
ax.set_ylabel("std (log scale)")
ax.set_title("on log-log axes, $\\sqrt{t}$ is a straight line of slope 1/2")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.show()

### Angle 2: the rescaling — every distribution is the *same* distribution in disguise

This is the most beautiful version of the √t fact, and the one that ties it to the **Central Limit Theorem**.

We saw in `02_` that the distribution of position at time $t$ is bell-shaped, with width that grows over time. The widths at $t = 50, 200, 1000$ are obviously different.

But what if we *divide* each position by $\sqrt{t}$? Since the typical scale of $X_t$ *is* $\sqrt{t}$, this rescaling cancels the time-dependent width. What's left should look the same at every $t$.

Specifically: $X_t / \sqrt{t}$ should approach a standard normal distribution $\mathcal{N}(0, 1)$ — the bell curve with mean 0 and std 1 — for all sufficiently large $t$.

In [ ]:
times_to_show = [50, 200, 1000]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

fig, ax = plt.subplots(figsize=(10, 5))
for t_val, color in zip(times_to_show, colors):
    rescaled = positions[:, t_val] / np.sqrt(t_val)
    ax.hist(rescaled, bins=50, alpha=0.45, density=True,
            color=color, label=f"$X_{{{t_val}}} / \\sqrt{{{t_val}}}$")

# overlay the standard normal density for comparison
x_grid = np.linspace(-4, 4, 300)
ax.plot(x_grid, np.exp(-x_grid**2 / 2) / np.sqrt(2 * np.pi),
        color="black", linewidth=2.5, label=r"$\mathcal{N}(0, 1)$")

ax.set_xlabel(r"$X_t / \sqrt{t}$")
ax.set_ylabel("density")
ax.set_title("rescaling by $1/\\sqrt{t}$ collapses every distribution onto the same bell curve")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

Three histograms, three different times, one shape. The black curve underneath is the standard normal density — the same one you'd get from $X_t / \sqrt{t}$ for any large $t$.

This is the deepest reading of the √t law: **$\sqrt{t}$ is the natural ruler for measuring random-walk distance**. Measured in those units, the walk's distribution is *time-invariant*.

### Angle 3: watch √t in real time

Animation of 200 walks fanning out, with the $\pm\sqrt{t}$ envelope drawn on top so you can see the prediction stay glued to the visible spread of paths.

In [ ]:
N_WALKS_ANIM = 200
positions_anim = positions[:N_WALKS_ANIM]

STEPS_PER_FRAME = 10
n_frames = N_STEPS // STEPS_PER_FRAME

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(0, N_STEPS)
ax.set_ylim(positions.min() - 5, positions.max() + 5)
ax.axhline(0, color="gray", linewidth=0.5, linestyle="--", alpha=0.5)
ax.set_xlabel("step number (t)")
ax.set_ylabel("position")
ax.set_title("$\\pm\\sqrt{t}$ envelope tracks the spread of the fan")
ax.grid(alpha=0.3)

walk_lines = [
    ax.plot([], [], linewidth=0.5, alpha=0.2, color="steelblue")[0]
    for _ in range(N_WALKS_ANIM)
]
(upper,) = ax.plot([], [], color="red", linewidth=2, label=r"$+\sqrt{t}$")
(lower,) = ax.plot([], [], color="red", linewidth=2, label=r"$-\sqrt{t}$")
ax.legend(loc="upper left")

def update(frame):
    end = (frame + 1) * STEPS_PER_FRAME + 1
    x_data = np.arange(end)
    for i, line in enumerate(walk_lines):
        line.set_data(x_data, positions_anim[i, :end])
    upper.set_data(x_data, np.sqrt(x_data))
    lower.set_data(x_data, -np.sqrt(x_data))
    return walk_lines + [upper, lower]

ani = FuncAnimation(fig, update, frames=n_frames, interval=40, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())

The red curves grow as $\sqrt{t}$ — slower than linear, faster than logarithmic. Roughly two-thirds of the visible walks stay inside the envelope at any given moment. (Exactly 68% if the distribution were perfectly Gaussian, which it nearly is for $t$ even moderately large.)

## What this means

The √t law sounds like a small technical fact — "standard deviation grows like square root of time." In practice it shapes how you think about anything driven by random fluctuation. A few flavors:

- **The leftover from cancellation is not zero, but it's not huge either.** Flip 1,000,000 fair coins, and the excess of heads over tails is *typically* about $\sqrt{1{,}000{,}000} = 1{,}000$ — not $0$, but not $1{,}000{,}000$ either. You'd see roughly 500,500 heads and 499,500 tails, give or take.
- **To double the typical distance traveled, you need four times as much time.** This is *very* slow growth. A walker that drifts ±10 in 100 steps will only drift ±100 in 10,000 steps.
- **The law is universal.** Our coin steps were ±1 with no special structure. But the same √t scaling holds for any independent steps with finite variance — Gaussian steps, dice rolls, weird-shaped distributions. Only the constant in front (the per-step σ) changes. The √t is forced by the math.

In the notebooks ahead we'll meet a 2D version (`04_`) and a multiplicative version (`05_`) where the rules change in interesting ways.

## Try this

- **Different step distribution.** Replace `rng.choice([-1, 1], size=...)` with `rng.normal(0, 1, size=...)` (Gaussian steps with variance 1). The std should *still* be exactly $\sqrt{t}$, because the per-step variance is still 1.
- **Larger steps.** Try `rng.choice([-2, 2], size=...)` (steps of size 2 instead of 1). The per-step variance is now 4. Predict: std should be $\sqrt{4t} = 2\sqrt{t}$. Check by re-running the std-vs-time plot.
- **Biased coin.** Use `rng.choice([-1, 1], size=..., p=[0.4, 0.6])`. Now there's a nonzero mean of $0.2$ per step, so the position drifts. The mean grows linearly ($0.2 \, t$) but the spread *around* the mean still grows as $\sqrt{t}$. Plot both.
- **Tiny `N_WALKS`.** Drop to 50 walks. The empirical std curve becomes wobbly — but the wobble is still centered on $\sqrt{t}$. Sampling noise vs underlying truth.

When you've poked at this enough, head to [`../04_two_dimensions/`](../04_two_dimensions/).